# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
import re
import asyncio
from playwright.async_api import async_playwright


In [2]:
project_path = Path.cwd()
project_data_sources_path = project_path / 'data' / 'sources'
project_data_exports_path = project_path / 'data' / 'exports'

print(project_path)
print(project_data_sources_path)
print(project_data_exports_path)

/Users/chrisizenour/Library/CloudStorage/Dropbox/phd/courses/ie_6683_machine_learning_with_industrial_engineering_applications/python_projects
/Users/chrisizenour/Library/CloudStorage/Dropbox/phd/courses/ie_6683_machine_learning_with_industrial_engineering_applications/python_projects/data/sources
/Users/chrisizenour/Library/CloudStorage/Dropbox/phd/courses/ie_6683_machine_learning_with_industrial_engineering_applications/python_projects/data/exports


# User Functions

## Box Score Dataset

In [3]:
def load_boxscore_dataset():
    """
    dataset originated from https://www.kaggle.com/datasets/cviaxmiwnptr/college-football-team-stats-2002-to-january-2024?resource=download
    download on 3/7/2026
    """
    df = pd.read_csv(
        project_data_sources_path / 'cfb_box_scores_2002_2025.csv',
        usecols=range(58),
        # sheet_name='Sheet1',
        # header=1,
        # engine='openpyxl',
    )
    # df = df.iloc[:-2]
    print(f'Shape of dataframe: {df.shape[0]} rows, {df.shape[1]} columns')
    return df

## Conference Map - Modern

In [4]:
# source: https://www.espn.com/college-football/teams
# accessed: 3/7/2026

conference_map = {
    'acc': ['boston college', 'california', 'clemson', 'duke', 'florida state', 'georgia tech', 'louisville','miami (fl)', 'nc state', 'north carolina', 'pittsburgh', 'smu', 'stanford','syracuse', 'virginia', 'virginia tech', 'wake forest'],

    'aac': ['army', 'charlotte', 'east carolina', 'florida atlantic', 'memphis', 'navy', 'north texas', 'rice', 'usf', 'temple', 'tulane', 'tulsa', 'uab', 'utsa'],

    'big12': ['arizona state', 'arizona', 'byu', 'baylor', 'cincinnati', 'colorado', 'houston', 'iowa state', 'kansas', 'kansas state', 'oklahoma', 'tcu', 'texas tech', 'ucf', 'utah', 'west virginia'],

    'big10': ['illinois', 'indiana', 'iowa', 'maryland', 'michigan state', 'michigan', 'minnesota','nebraska', 'northwestern', 'ohio state', 'oregon', 'penn state', 'purdue', 'rutgers', 'ucla', 'usc', 'washington', 'wisconsin'],

    'cusa': ['delaware', 'fiu', 'jacksonville state', 'kennesaw state', 'liberty', 'louisiana tech', 'middle tennessee', 'missouri state', 'new mexico state','sam houston', 'western kentucky'],

    'ind': ['notre dame', 'uconn'],

    'mwc': ['air force', 'hawaii', 'nevada', 'new mexico', 'north dakota state', 'northern illinois', 'san jose state', 'unlv', 'utep', 'wyoming'],

    'pac12': ['boise state', 'colorado state', 'fresno state', 'oregon state', 'san diego state', 'texas state', 'utah state', 'washington state'],

    'sec': ['alabama', 'arkansas', 'auburn', 'florida', 'georgia', 'kentucky', 'lsu', 'mississippi state', 'missouri', 'oklahoma', 'ole miss', 'south carolina', 'tennessee', 'texas a&m', 'texas', 'vanderbilt'],

    'sun-belt': ['appalachian state', 'arkansas state', 'coastal carolina', 'georgia southern', 'georgia state', 'james madison', 'louisiana', 'marshall', 'old dominion', 'south alabama', 'southern miss', 'troy', 'ul monroe'],

    'mac': ['akron', 'ball state', 'bowling green', 'buffalo', 'central michigan', 'eastern michigan', 'kent state', 'massachusetts', 'miami (oh)', 'ohio', 'sacramento state', 'toledo', 'western michigan']
}

## 247 Composite Team Rankings

In [5]:
async def scrape_single_247_page(page, year, ranking_type):
    row_locator = page.locator("li.rankings-page__list-item")
    row_count = await row_locator.count()

    rows = []

    for i in range(row_count):
        row = row_locator.nth(i)

        team_loc = row.locator("div.team")
        avg_loc = row.locator("div.avg")
        points_loc = row.locator("div.points a.number")

        if await team_loc.count() == 0:
            continue
        if await avg_loc.count() == 0:
            continue
        if await points_loc.count() == 0:
            continue

        team = (await team_loc.first.inner_text()).strip()
        avg = (await avg_loc.first.inner_text()).strip()
        points = (await points_loc.first.inner_text()).strip()

        if not team or team.lower() == "team":
            continue

        rows.append({
            "year": year,
            "team": team,
            "ranking_type": ranking_type,
            "avg": avg,
            "points": points,
        })

    return rows

In [6]:
async def scrape_category_url(page, url, year, ranking_type):
    await page.goto(url, wait_until="domcontentloaded", timeout=90000)
    await page.wait_for_timeout(2500)

    body_text = await page.locator("body").inner_text()
    if "No Results" in body_text:
        return []

    while True:
        load_more = page.locator("text=Load More")

        if await load_more.count() == 0:
            break

        try:
            await load_more.first.scroll_into_view_if_needed()
            await page.wait_for_timeout(750)
            await load_more.first.click(timeout=10000)
            await page.wait_for_timeout(2000)
        except Exception:
            break

    rows = await scrape_single_247_page(page, year, ranking_type)
    return rows

In [7]:
async def scrape_247_rankings_2009_2030(headless=True):
    all_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1400, "height": 2200},
        )
        page = await context.new_page()

        for year in range(2009, 2031):
            print(f"Scraping {year}...")

            # Recruiting composite exists across all years
            recruit_url = f"https://247sports.com/season/{year}-football/compositeteamrankings/"
            recruit_rows = await scrape_category_url(
                page=page,
                url=recruit_url,
                year=year,
                ranking_type="recruiting"
            )
            print(f"  recruiting rows: {len(recruit_rows)}")
            all_rows.extend(recruit_rows)

            # Overall and transfer start in 2022
            if year >= 2022:
                overall_url = f"https://247sports.com/season/{year}-football/overallteamrankings/"
                overall_rows = await scrape_category_url(
                    page=page,
                    url=overall_url,
                    year=year,
                    ranking_type="overall"
                )
                print(f"  overall rows: {len(overall_rows)}")
                all_rows.extend(overall_rows)

                transfer_url = f"https://247sports.com/season/{year}-football/transferteamrankings/"
                transfer_rows = await scrape_category_url(
                    page=page,
                    url=transfer_url,
                    year=year,
                    ranking_type="transfer"
                )
                print(f"  transfer rows: {len(transfer_rows)}")
                all_rows.extend(transfer_rows)

        await browser.close()

    df_long = pd.DataFrame(all_rows)

    if not df_long.empty:
        df_long["avg"] = pd.to_numeric(df_long["avg"], errors="coerce")
        df_long["points"] = (
            df_long["points"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df_long["points"] = pd.to_numeric(df_long["points"], errors="coerce")

        df_long = df_long.drop_duplicates().reset_index(drop=True)

    return df_long

In [8]:
def reshape_247_rankings_wide(df_long):
    df_long = df_long.copy()

    df_wide = (
        df_long
        .pivot_table(
            index=["year", "team"],
            columns="ranking_type",
            values=["avg", "points"],
            aggfunc="first"
        )
    )

    df_wide.columns = [f"{ranking_type}_{metric}" for metric, ranking_type in df_wide.columns]
    df_wide = df_wide.reset_index()

    desired_cols = [
        "year",
        "team",
        "recruiting_avg",
        "recruiting_points",
        "overall_avg",
        "overall_points",
        "transfer_avg",
        "transfer_points",
    ]

    for col in desired_cols:
        if col not in df_wide.columns:
            df_wide[col] = pd.NA

    df_wide = df_wide[desired_cols]

    return df_wide

In [9]:
# original approach not accounting for overall, recruit or transfer
async def scrape_247_team_rankings(start_year=2009, end_year=2030, headless=True):
    all_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1400, "height": 2200},
        )
        page = await context.new_page()

        for year in range(start_year, end_year + 1):
            print(f"Scraping {year}...")
            url = f"https://247sports.com/season/{year}-football/compositeteamrankings/"
            await page.goto(url, wait_until="domcontentloaded", timeout=90000)
            await page.wait_for_timeout(2500)

            body_text = await page.locator("body").inner_text()
            if f"No Results for {year} Football" in body_text:
                print(f"  No results for {year}")
                continue

            while True:
                load_more = page.locator("text=Load More")
                if await load_more.count() == 0:
                    break

                try:
                    await load_more.first.scroll_into_view_if_needed()
                    await page.wait_for_timeout(750)
                    await load_more.first.click(timeout=10000)
                    await page.wait_for_timeout(2000)
                except Exception:
                    break

            row_locator = page.locator("li.rankings-page__list-item")
            row_count = await row_locator.count()

            year_rows = 0

            for i in range(row_count):
                row = row_locator.nth(i)

                team_loc = row.locator("div.team")
                avg_loc = row.locator("div.avg")
                points_loc = row.locator("div.points a.number")

                if await team_loc.count() == 0:
                    continue
                if await avg_loc.count() == 0:
                    continue
                if await points_loc.count() == 0:
                    continue

                team = (await team_loc.first.inner_text()).strip()
                avg = (await avg_loc.first.inner_text()).strip()
                points = (await points_loc.first.inner_text()).strip()

                if not team or team.lower() == "team":
                    continue

                all_rows.append({
                    "year": year,
                    "team": team,
                    "avg": avg,
                    "points": points,
                })
                year_rows += 1

            print(f"  rows found: {year_rows}")

        await browser.close()

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df["avg"] = pd.to_numeric(df["avg"], errors="coerce")
        df["points"] = (
            df["points"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df["points"] = pd.to_numeric(df["points"], errors="coerce")
        df = df.drop_duplicates().reset_index(drop=True)

    return df

## General Use

In [10]:
def clean_df_columns(df):
    df = df.copy()
    df.columns = df.columns.str.lower()
    df.columns = df.columns.str.strip()
    df.columns = df.columns.str.replace(' ', '_', regex=False)
    df.columns = df.columns.str.replace(r'_+', '_', regex=True)
    return df

In [11]:
def set_date_cols_as_datetime(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], format='%m/%d/%y', errors='coerce')
    return df

In [12]:
def create_season_column(df):
    df = df.copy()
    df['season'] = np.where(
        df['date'].dt.month == 1,
        df['date'].dt.year - 1,
        df['date'].dt.year
    )
    return df

## Filtering Down Box Score Dataset

In [13]:
def retain_specified_seasons(df, initial_season, final_season):
    df = df.copy()
    df = df.loc[df['season'].between(initial_season, final_season)]
    return df

In [14]:
def retain_regular_season_games(df):
    df = df.copy()
    army_navy_mask = (
        df['away'].astype(str).str.strip().str.lower().isin(['army', 'navy']) &
        df['home'].astype(str).str.strip().str.lower().isin(['army', 'navy'])
    )

    regular_mask = df['game_type'].astype(str).str.strip().str.lower().eq('regular')

    df = df.loc[regular_mask | army_navy_mask]
    return df

In [15]:
def find_unmapped_teams(df, conference_map):
    df = df.copy()

    valid_teams = {
        team.strip().lower()
        for teams in conference_map.values()
        for team in teams
    }

    home_teams = (
        df['home']
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

    away_teams = (
        df['away']
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

    all_teams_in_data = set(home_teams).union(set(away_teams))
    unmapped_teams = sorted(all_teams_in_data - valid_teams)

    return unmapped_teams

In [16]:
def retain_games_for_specified_conferences(df, conference_map, conferences_to_keep, both_teams=True, additional_teams=None):
    df = df.copy()
    conferences_to_keep = [conf.lower() for conf in conferences_to_keep]

    teams_to_keep = {
        team.strip().lower()
        for conf, teams in conference_map.items()
        if conf.lower() in conferences_to_keep
        for team in teams
    }

    if additional_teams is not None:
        extra_teams = {team.strip().lower() for team in additional_teams}
        teams_to_keep = teams_to_keep.union(extra_teams)

    home_mask = (
        df['home']
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(teams_to_keep)
    )

    away_mask = (
        df['away']
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(teams_to_keep)
    )

    if both_teams:
        df = df.loc[home_mask & away_mask]
    else:
        df = df.loc[home_mask | away_mask]

    return df

## Compare Team Names of Box Score and 247Sports

In [17]:
def normalize_team_name(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


In [18]:
def compare_team_names(box_score_df, df_247_wide):
    box_score_df = box_score_df.copy()
    df_247_wide = df_247_wide.copy()

    box_home_teams = set(normalize_team_name(box_score_df['home']).dropna().unique())
    box_away_teams = set(normalize_team_name(box_score_df['away']).dropna().unique())
    box_all_teams = box_home_teams.union(box_away_teams)

    sports247_teams = set(normalize_team_name(df_247_wide['team']).dropna().unique())

    only_in_box_score = sorted(box_all_teams - sports247_teams)
    only_in_247 = sorted(sports247_teams - box_all_teams)
    in_both = sorted(box_all_teams.intersection(sports247_teams))

    comparison_dict = {
        'only_in_box_score': pd.DataFrame({'team': only_in_box_score}),
        'only_in_247': pd.DataFrame({'team': only_in_247}),
        'in_both': pd.DataFrame({'team': in_both}),
    }

    return comparison_dict

In [19]:
def build_team_season_247_summary_table(df_247_wide):
    df = df_247_wide.copy()

    df = df[
        [
            'year',
            'team',
            'recruiting_avg',
            'recruiting_points',
            'overall_avg',
            'overall_points',
            'transfer_avg',
            'transfer_points',
        ]
    ].copy()

    summary_rows = []

    min_year = int(df['year'].min())
    max_year = int(df['year'].max())

    for season in range(min_year, max_year + 1):
        season_rows = []

        freshman_year = season
        sophomore_year = season - 1
        junior_year = season - 2
        senior_year = season - 3

        class_year_map = {
            'freshman': freshman_year,
            'sophomore': sophomore_year,
            'junior': junior_year,
            'senior': senior_year,
        }

        class_years = list(class_year_map.values())

        teams_in_window = sorted(df.loc[df['year'].isin(class_years), 'team'].dropna().unique())

        for team in teams_in_window:
            team_class_df = df.loc[
                (df['team'] == team) & (df['year'].isin(class_years)),
                ['year', 'recruiting_avg', 'recruiting_points']
            ].copy()

            freshman_row = team_class_df.loc[team_class_df['year'] == freshman_year]
            sophomore_row = team_class_df.loc[team_class_df['year'] == sophomore_year]
            junior_row = team_class_df.loc[team_class_df['year'] == junior_year]
            senior_row = team_class_df.loc[team_class_df['year'] == senior_year]

            same_season_row = df.loc[
                (df['team'] == team) & (df['year'] == season),
                ['overall_avg', 'overall_points', 'transfer_avg', 'transfer_points']
            ]

            recruiting_avg_values = team_class_df['recruiting_avg'].dropna()
            recruiting_points_values = team_class_df['recruiting_points'].dropna()

            row = {
                'season': season,
                'team': team,

                'freshman_class_year': freshman_year,
                'sophomore_class_year': sophomore_year,
                'junior_class_year': junior_year,
                'senior_class_year': senior_year,

                'freshman_recruiting_avg': freshman_row['recruiting_avg'].iloc[0] if not freshman_row.empty else pd.NA,
                'freshman_recruiting_points': freshman_row['recruiting_points'].iloc[0] if not freshman_row.empty else pd.NA,

                'sophomore_recruiting_avg': sophomore_row['recruiting_avg'].iloc[0] if not sophomore_row.empty else pd.NA,
                'sophomore_recruiting_points': sophomore_row['recruiting_points'].iloc[0] if not sophomore_row.empty else pd.NA,

                'junior_recruiting_avg': junior_row['recruiting_avg'].iloc[0] if not junior_row.empty else pd.NA,
                'junior_recruiting_points': junior_row['recruiting_points'].iloc[0] if not junior_row.empty else pd.NA,

                'senior_recruiting_avg': senior_row['recruiting_avg'].iloc[0] if not senior_row.empty else pd.NA,
                'senior_recruiting_points': senior_row['recruiting_points'].iloc[0] if not senior_row.empty else pd.NA,

                'recruiting_avg_mean_4yr': recruiting_avg_values.mean() if not recruiting_avg_values.empty else pd.NA,
                'recruiting_avg_sum_4yr': recruiting_avg_values.sum() if not recruiting_avg_values.empty else pd.NA,
                'recruiting_points_mean_4yr': recruiting_points_values.mean() if not recruiting_points_values.empty else pd.NA,
                'recruiting_points_sum_4yr': recruiting_points_values.sum() if not recruiting_points_values.empty else pd.NA,
                'recruiting_class_count': int(team_class_df['year'].nunique()),
                'recruiting_class_year_min': team_class_df['year'].min() if not team_class_df.empty else pd.NA,
                'recruiting_class_year_max': team_class_df['year'].max() if not team_class_df.empty else pd.NA,

                'overall_avg': same_season_row['overall_avg'].iloc[0] if not same_season_row.empty else pd.NA,
                'overall_points': same_season_row['overall_points'].iloc[0] if not same_season_row.empty else pd.NA,
                'transfer_avg': same_season_row['transfer_avg'].iloc[0] if not same_season_row.empty else pd.NA,
                'transfer_points': same_season_row['transfer_points'].iloc[0] if not same_season_row.empty else pd.NA,
            }

            season_rows.append(row)

        summary_rows.extend(season_rows)

    team_season_247_summary_df = pd.DataFrame(summary_rows)

    return team_season_247_summary_df[
        [
            'season',
            'team',

            'freshman_class_year',
            'freshman_recruiting_avg',
            'freshman_recruiting_points',

            'sophomore_class_year',
            'sophomore_recruiting_avg',
            'sophomore_recruiting_points',

            'junior_class_year',
            'junior_recruiting_avg',
            'junior_recruiting_points',

            'senior_class_year',
            'senior_recruiting_avg',
            'senior_recruiting_points',

            'recruiting_avg_mean_4yr',
            'recruiting_avg_sum_4yr',
            'recruiting_points_mean_4yr',
            'recruiting_points_sum_4yr',
            'recruiting_class_count',
            'recruiting_class_year_min',
            'recruiting_class_year_max',

            'overall_avg',
            'overall_points',
            'transfer_avg',
            'transfer_points',
        ]
    ]

In [20]:
def merge_247_summary_onto_box_scores(box_score_df, team_season_247_summary_df):
    df_box = box_score_df.copy()
    df_247 = team_season_247_summary_df.copy()

    home_247 = df_247.rename(columns={
        'team': 'home',
        'freshman_class_year': 'home_freshman_class_year',
        'sophomore_class_year': 'home_sophomore_class_year',
        'junior_class_year': 'home_junior_class_year',
        'senior_class_year': 'home_senior_class_year',
        'recruiting_avg_mean_4yr': 'home_recruiting_avg_mean_4yr',
        'recruiting_points_mean_4yr': 'home_recruiting_points_mean_4yr',
        'recruiting_points_sum_4yr': 'home_recruiting_points_sum_4yr',
        'recruiting_class_count': 'home_recruiting_class_count',
        'recruiting_class_year_min': 'home_recruiting_class_year_min',
        'recruiting_class_year_max': 'home_recruiting_class_year_max',
        'overall_avg': 'home_overall_avg',
        'overall_points': 'home_overall_points',
        'transfer_avg': 'home_transfer_avg',
        'transfer_points': 'home_transfer_points',
    })

    away_247 = df_247.rename(columns={
        'team': 'away',
        'freshman_class_year': 'away_freshman_class_year',
        'sophomore_class_year': 'away_sophomore_class_year',
        'junior_class_year': 'away_junior_class_year',
        'senior_class_year': 'away_senior_class_year',
        'recruiting_avg_mean_4yr': 'away_recruiting_avg_mean_4yr',
        'recruiting_points_mean_4yr': 'away_recruiting_points_mean_4yr',
        'recruiting_points_sum_4yr': 'away_recruiting_points_sum_4yr',
        'recruiting_class_count': 'away_recruiting_class_count',
        'recruiting_class_year_min': 'away_recruiting_class_year_min',
        'recruiting_class_year_max': 'away_recruiting_class_year_max',
        'overall_avg': 'away_overall_avg',
        'overall_points': 'away_overall_points',
        'transfer_avg': 'away_transfer_avg',
        'transfer_points': 'away_transfer_points',
    })

    df_box = df_box.merge(home_247, how='left', on=['season', 'home'])
    df_box = df_box.merge(away_247, how='left', on=['season', 'away'])

    return df_box

In [21]:
def build_team_season_recruiting_detail_table(df_247_wide):
    df = df_247_wide.copy()

    df = df[['year', 'team', 'recruiting_avg', 'recruiting_points']].copy()

    detail_rows = []

    min_year = int(df['year'].min())
    max_year = int(df['year'].max())

    for season in range(min_year, max_year + 1):
        class_years = [season - 3, season - 2, season - 1, season]

        season_df = df.loc[df['year'].isin(class_years)].copy()
        season_df['season'] = season
        season_df = season_df.rename(columns={
            'year': 'recruiting_class_year',
            'recruiting_avg': 'class_recruiting_avg',
            'recruiting_points': 'class_recruiting_points'
        })

        detail_rows.append(season_df)

    team_season_recruiting_detail_df = pd.concat(detail_rows, ignore_index=True)

    return team_season_recruiting_detail_df[
        ['season', 'team', 'recruiting_class_year', 'class_recruiting_avg', 'class_recruiting_points']
    ]

In [22]:
def safe_divide(num, den):
    return np.where((den.isna()) | (den == 0), np.nan, num / den)


# Data Imports

Scrape data from 247. Run once to store data in data sources folder. Thereafter access the 247 recruiting data from the via the sources folder

In [23]:
# last data scrape occurred on 3/21/26 at ~730 am

# df_247_long = await scrape_247_rankings_2009_2030(headless=True)
# df_247_long.to_csv(project_data_sources_path / "247sports_rankings_2009_2030_long.csv", index=False)
# df_247_long = pd.read_csv(project_data_sources_path / "247sports_rankings_2009_2030_long.csv")
# df_247_long

In [24]:
df_247_long = pd.read_csv(project_data_sources_path / "247sports_rankings_2009_2030_long.csv")
df_247_long


,year,team,ranking_type,avg,points
0,2009,LSU,recruiting,91.51,293.15
1,2009,USC,recruiting,93.59,289.28
2,2009,Alabama,recruiting,91.39,288.63
3,2009,Georgia,recruiting,91.81,283.72
4,2009,Ohio State,recruiting,91.28,281.93
...,...,...,...,...,...
5537,2028,Oklahoma,overall,97.90,27.90
5538,2028,Penn State,overall,89.00,19.00
5539,2028,Purdue,overall,88.50,18.50
5540,2028,Boston College,overall,88.00,18.00


In [25]:
df_247_long['team'] = df_247_long['team'].str.strip().str.lower()
df_247_long_team_name_map = {
    'miami': 'miami (fl)',
    'connecticut': 'uconn',
    'louisiana-monroe': 'ul monroe',
    'south florida': 'usf',
    'umass': 'massachusetts',
}
df_247_long['team'] = df_247_long['team'].replace(df_247_long_team_name_map)

df_247_wide = reshape_247_rankings_wide(df_247_long)
team_season_247_summary_df = build_team_season_247_summary_table(df_247_wide)

team_season_247_summary_df

,season,team,freshman_class_year,freshman_recruiting_avg,freshman_recruiting_points,sophomore_class_year,sophomore_recruiting_avg,sophomore_recruiting_points,junior_class_year,junior_recruiting_avg,...,recruiting_avg_sum_4yr,recruiting_points_mean_4yr,recruiting_points_sum_4yr,recruiting_class_count,recruiting_class_year_min,recruiting_class_year_max,overall_avg,overall_points,transfer_avg,transfer_points
0,2009,air force,2009,77.06,95.62,2008,<NA>,<NA>,2007,<NA>,...,77.06,95.62,95.62,1,2009,2009,NaN,NaN,NaN,NaN
1,2009,akron,2009,78.25,124.68,2008,<NA>,<NA>,2007,<NA>,...,78.25,124.68,124.68,1,2009,2009,NaN,NaN,NaN,NaN
2,2009,alabama,2009,91.39,288.63,2008,<NA>,<NA>,2007,<NA>,...,91.39,288.63,288.63,1,2009,2009,NaN,NaN,NaN,NaN
3,2009,alabama state,2009,76.67,6.67,2008,<NA>,<NA>,2007,<NA>,...,76.67,6.67,6.67,1,2009,2009,NaN,NaN,NaN,NaN
4,2009,arizona,2009,84.23,184.83,2008,<NA>,<NA>,2007,<NA>,...,84.23,184.83,184.83,1,2009,2009,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4743,2028,wisconsin,2028,<NA>,<NA>,2027,88.6,72.98,2026,87.49,...,264.49,162.783333,488.35,3,2025,2027,<NA>,<NA>,<NA>,<NA>
4744,2028,wofford,2028,<NA>,<NA>,2027,<NA>,<NA>,2026,85.44,...,85.44,60.53,60.53,2,2025,2026,<NA>,<NA>,<NA>,<NA>
4745,2028,wyoming,2028,<NA>,<NA>,2027,84.0,14.0,2026,83.36,...,249.99,115.653333,346.96,3,2025,2027,<NA>,<NA>,<NA>,<NA>
4746,2028,yale,2028,<NA>,<NA>,2027,<NA>,<NA>,2026,83.0,...,164.02,47.31,94.62,2,2025,2026,<NA>,<NA>,<NA>,<NA>


In [26]:
box_score_df = load_boxscore_dataset()
box_score_df = clean_df_columns(box_score_df)

box_score_df['home'] = normalize_team_name(box_score_df['home'])
box_score_df['away'] = normalize_team_name(box_score_df['away'])

box_score_df = set_date_cols_as_datetime(box_score_df)
box_score_df = create_season_column(box_score_df)
box_score_df = merge_247_summary_onto_box_scores(box_score_df, team_season_247_summary_df)
# box_score_df.to_csv(project_data_exports_path / "box_score_df.csv", index=False)
box_score_df = retain_specified_seasons(box_score_df, 2014, 2025)
box_score_df = retain_regular_season_games(box_score_df)
box_score_df = box_score_df.drop(['game_type'], axis=1)
power_conf_df = retain_games_for_specified_conferences(
    box_score_df,
    conference_map,
    ['acc', 'big12', 'big10', 'sec'],
    both_teams=True,
    additional_teams=['notre dame', 'washington state', 'oregon state']
).reset_index(drop=True)
power_conf_df.to_csv(project_data_exports_path / "power_conf_df.csv", index=False)

Shape of dataframe: 19843 rows, 58 columns


In [27]:
power_conf_df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,recruiting_avg_sum_4yr_y,away_recruiting_points_mean_4yr,away_recruiting_points_sum_4yr,away_recruiting_class_count,away_recruiting_class_year_min,away_recruiting_class_year_max,away_overall_avg,away_overall_points,away_transfer_avg,away_transfer_points
0,2014,1.0,2014-08-28,6:00 PM,texas a&m,south carolina,21.0,9.0,sec,sec,...,353.7,246.985,987.94,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
1,2014,1.0,2014-08-28,10:00 PM,rutgers,washington state,NaN,NaN,big10,pac12,...,337.95,196.7625,787.05,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
2,2014,1.0,2014-08-30,8:30 AM,penn state,ucf,NaN,NaN,big10,aac,...,344.98,202.77,811.08,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
3,2014,1.0,2014-08-30,12:00 PM,ucla,virginia,7.0,NaN,pac12,acc,...,349.93,234.0175,936.07,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
4,2014,1.0,2014-08-30,3:30 PM,west virginia,alabama,NaN,2.0,big12,sec,...,335.57,190.7225,762.89,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,10:30 PM,notre dame,stanford,9.0,NaN,ind,acc,...,366.07,273.49,1093.96,4.0,2022.0,2025.0,90.15,274.65,87.14,24.48
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,341.77,194.9975,779.99,4.0,2022.0,2025.0,85.65,211.17,84.31,17.82
3820,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,373.47,313.96,1255.84,4.0,2022.0,2025.0,91.81,308.3,88.5,40.15
3821,2025,15.0,2025-12-06,8:00 PM,indiana,ohio state,2.0,1.0,big10,big10,...,346.58,194.9925,779.97,4.0,2022.0,2025.0,86.51,223.04,86.43,38.68


In [28]:
power_conf_df['season'].value_counts(dropna=False)

season
2023    332
2025    331
2024    328
2021    326
2016    324
2022    323
2017    322
2018    322
2019    321
2014    316
2015    316
2020    262
Name: count, dtype: int64

In [29]:
power_conf_df['conf_away'].value_counts(dropna=False)

conf_away
big10    857
acc      850
sec      789
pac12    599
big12    580
ind       76
aac       72
Name: count, dtype: int64

In [30]:
power_conf_df['conf_home'].value_counts(dropna=False)

conf_home
big10    865
acc      846
sec      810
pac12    592
big12    577
ind       69
aac       64
Name: count, dtype: int64

# Power Conf DF -> Model Table

In [31]:
df = pd.read_csv(project_data_exports_path / "power_conf_df.csv")
df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,recruiting_avg_sum_4yr_y,away_recruiting_points_mean_4yr,away_recruiting_points_sum_4yr,away_recruiting_class_count,away_recruiting_class_year_min,away_recruiting_class_year_max,away_overall_avg,away_overall_points,away_transfer_avg,away_transfer_points
0,2014,1.0,2014-08-28,6:00 PM,texas a&m,south carolina,21.0,9.0,sec,sec,...,353.70,246.9850,987.94,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
1,2014,1.0,2014-08-28,10:00 PM,rutgers,washington state,NaN,NaN,big10,pac12,...,337.95,196.7625,787.05,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
2,2014,1.0,2014-08-30,8:30 AM,penn state,ucf,NaN,NaN,big10,aac,...,344.98,202.7700,811.08,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
3,2014,1.0,2014-08-30,12:00 PM,ucla,virginia,7.0,NaN,pac12,acc,...,349.93,234.0175,936.07,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
4,2014,1.0,2014-08-30,3:30 PM,west virginia,alabama,NaN,2.0,big12,sec,...,335.57,190.7225,762.89,4.0,2011.0,2014.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,10:30 PM,notre dame,stanford,9.0,NaN,ind,acc,...,366.07,273.4900,1093.96,4.0,2022.0,2025.0,90.15,274.65,87.14,24.48
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,341.77,194.9975,779.99,4.0,2022.0,2025.0,85.65,211.17,84.31,17.82
3820,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,373.47,313.9600,1255.84,4.0,2022.0,2025.0,91.81,308.30,88.50,40.15
3821,2025,15.0,2025-12-06,8:00 PM,indiana,ohio state,2.0,1.0,big10,big10,...,346.58,194.9925,779.97,4.0,2022.0,2025.0,86.51,223.04,86.43,38.68


# Create Modeling Dataframe

## Load Power Conf DF and Filter

In [32]:
df = df.loc[df['season'].between(2014, 2025), :].copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')

text_cols = ['away', 'home', 'conf_away', 'conf_home', 'tv']
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.lower()

df['neutral'] = df['neutral'].fillna(False).astype(int)

df = df.sort_values(['season', 'week', 'date', 'away', 'home']).reset_index(drop=True)
df['game_id'] = np.arange(1, len(df)+1)

df['margin_home'] = df['score_home'] - df['score_away']
df['home_win'] = (df['margin_home'] > 0).astype(int)

df['conference_game'] = (df['conf_home'] == df['conf_away']).astype(int)
df['ranked_home'] = df['rank_home'].notna().astype(int)
df['ranked_away'] = df['rank_away'].notna().astype(int)
df['both_ranked'] = ((df['rank_home'].notna()) & (df['rank_away'].notna())).astype(int)

rank_fallback = 999
df['rank_home_filled'] = df['rank_home'].fillna(rank_fallback)
df['rank_away_filled'] = df['rank_away'].fillna(rank_fallback)
df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,away_transfer_points,game_id,margin_home,home_win,conference_game,ranked_home,ranked_away,both_ranked,rank_home_filled,rank_away_filled
0,2014,1.0,2014-08-28,10:00 PM,rutgers,washington state,NaN,NaN,big10,pac12,...,NaN,1,-3,0,0,0,0,0,999.0,999.0
1,2014,1.0,2014-08-28,6:00 PM,texas a&m,south carolina,21.0,9.0,sec,sec,...,NaN,2,-24,0,1,1,1,1,9.0,21.0
2,2014,1.0,2014-08-30,4:00 PM,arkansas,auburn,NaN,6.0,sec,sec,...,NaN,3,24,1,1,1,0,0,6.0,999.0
3,2014,1.0,2014-08-30,3:30 PM,california,northwestern,NaN,NaN,pac12,big10,...,NaN,4,-7,0,0,0,0,0,999.0,999.0
4,2014,1.0,2014-08-30,5:30 PM,clemson,georgia,16.0,12.0,acc,sec,...,NaN,5,24,1,0,1,1,1,12.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,3:30 PM,wisconsin,minnesota,NaN,NaN,big10,big10,...,47.40,3819,10,1,1,0,0,0,999.0,999.0
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,17.82,3820,27,1,1,1,1,1,4.0,11.0
3820,2025,15.0,2025-12-06,8:00 PM,duke,virginia,NaN,17.0,acc,acc,...,22.62,3821,-7,0,1,1,0,0,17.0,999.0
3821,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,40.15,3822,-21,0,1,1,1,1,9.0,3.0


## Create Home Team Game Long Table
One row per team per game

In [33]:
home_df = df[[
    "game_id", "season", "week", "date",
    "home", "away", "conf_home", "conf_away", "neutral",
    "score_home", "score_away",
    "first_downs_home", "first_downs_away",
    "third_down_comp_home", "third_down_att_home",
    "third_down_comp_away", "third_down_att_away",
    "fourth_down_comp_home", "fourth_down_att_home",
    "fourth_down_comp_away", "fourth_down_att_away",
    "pass_comp_home", "pass_att_home", "pass_yards_home",
    "pass_comp_away", "pass_att_away", "pass_yards_away",
    "rush_att_home", "rush_yards_home",
    "rush_att_away", "rush_yards_away",
    "total_yards_home", "total_yards_away",
    "fum_home", "fum_away",
    "int_home", "int_away",
    "pen_num_home", "pen_yards_home",
    "pen_num_away", "pen_yards_away",
    "possession_home", "possession_away",
    "rank_home", "rank_away",
    "home_recruiting_avg_mean_4yr", "home_recruiting_points_mean_4yr",
    "home_recruiting_points_sum_4yr", "home_overall_avg", "home_overall_points",
    "home_transfer_avg", "home_transfer_points"
]].copy()

home_df = home_df.rename(columns={
    "home": "team",
    "away": "opponent",
    "conf_home": "team_conf",
    "conf_away": "opp_conf",
    "score_home": "points_for",
    "score_away": "points_against",
    "first_downs_home": "first_downs_for",
    "first_downs_away": "first_downs_against",
    "third_down_comp_home": "third_down_comp_for",
    "third_down_att_home": "third_down_att_for",
    "third_down_comp_away": "third_down_comp_against",
    "third_down_att_away": "third_down_att_against",
    "fourth_down_comp_home": "fourth_down_comp_for",
    "fourth_down_att_home": "fourth_down_att_for",
    "fourth_down_comp_away": "fourth_down_comp_against",
    "fourth_down_att_away": "fourth_down_att_against",
    "pass_comp_home": "pass_comp_for",
    "pass_att_home": "pass_att_for",
    "pass_yards_home": "pass_yards_for",
    "pass_comp_away": "pass_comp_against",
    "pass_att_away": "pass_att_against",
    "pass_yards_away": "pass_yards_against",
    "rush_att_home": "rush_att_for",
    "rush_yards_home": "rush_yards_for",
    "rush_att_away": "rush_att_against",
    "rush_yards_away": "rush_yards_against",
    "total_yards_home": "total_yards_for",
    "total_yards_away": "total_yards_against",
    "fum_home": "fum_for",
    "fum_away": "fum_against",
    "int_home": "int_for",
    "int_away": "int_against",
    "pen_num_home": "pen_num_for",
    "pen_yards_home": "pen_yards_for",
    "pen_num_away": "pen_num_against",
    "pen_yards_away": "pen_yards_against",
    "possession_home": "possession_for",
    "possession_away": "possession_against",
    "rank_home": "team_rank",
    "rank_away": "opp_rank",
    "home_recruiting_avg_mean_4yr": "recruiting_avg_mean_4yr",
    "home_recruiting_points_mean_4yr": "recruiting_points_mean_4yr",
    "home_recruiting_points_sum_4yr": "recruiting_points_sum_4yr",
    "home_overall_avg": "overall_avg",
    "home_overall_points": "overall_points",
    "home_transfer_avg": "transfer_avg",
    "home_transfer_points": "transfer_points"
})
home_df["is_home"] = 1

## Create Away Team Game Long Table
One row per team per game

In [34]:
away_df = df[[
    "game_id", "season", "week", "date",
    "away", "home", "conf_away", "conf_home", "neutral",
    "score_away", "score_home",
    "first_downs_away", "first_downs_home",
    "third_down_comp_away", "third_down_att_away",
    "third_down_comp_home", "third_down_att_home",
    "fourth_down_comp_away", "fourth_down_att_away",
    "fourth_down_comp_home", "fourth_down_att_home",
    "pass_comp_away", "pass_att_away", "pass_yards_away",
    "pass_comp_home", "pass_att_home", "pass_yards_home",
    "rush_att_away", "rush_yards_away",
    "rush_att_home", "rush_yards_home",
    "total_yards_away", "total_yards_home",
    "fum_away", "fum_home",
    "int_away", "int_home",
    "pen_num_away", "pen_yards_away",
    "pen_num_home", "pen_yards_home",
    "possession_away", "possession_home",
    "rank_away", "rank_home",
    "away_recruiting_avg_mean_4yr", "away_recruiting_points_mean_4yr",
    "away_recruiting_points_sum_4yr", "away_overall_avg", "away_overall_points",
    "away_transfer_avg", "away_transfer_points"
]].copy()

away_df = away_df.rename(columns={
    "away": "team",
    "home": "opponent",
    "conf_away": "team_conf",
    "conf_home": "opp_conf",
    "score_away": "points_for",
    "score_home": "points_against",
    "first_downs_away": "first_downs_for",
    "first_downs_home": "first_downs_against",
    "third_down_comp_away": "third_down_comp_for",
    "third_down_att_away": "third_down_att_for",
    "third_down_comp_home": "third_down_comp_against",
    "third_down_att_home": "third_down_att_against",
    "fourth_down_comp_away": "fourth_down_comp_for",
    "fourth_down_att_away": "fourth_down_att_for",
    "fourth_down_comp_home": "fourth_down_comp_against",
    "fourth_down_att_home": "fourth_down_att_against",
    "pass_comp_away": "pass_comp_for",
    "pass_att_away": "pass_att_for",
    "pass_yards_away": "pass_yards_for",
    "pass_comp_home": "pass_comp_against",
    "pass_att_home": "pass_att_against",
    "pass_yards_home": "pass_yards_against",
    "rush_att_away": "rush_att_for",
    "rush_yards_away": "rush_yards_for",
    "rush_att_home": "rush_att_against",
    "rush_yards_home": "rush_yards_against",
    "total_yards_away": "total_yards_for",
    "total_yards_home": "total_yards_against",
    "fum_away": "fum_for",
    "fum_home": "fum_against",
    "int_away": "int_for",
    "int_home": "int_against",
    "pen_num_away": "pen_num_for",
    "pen_yards_away": "pen_yards_for",
    "pen_num_home": "pen_num_against",
    "pen_yards_home": "pen_yards_against",
    "possession_away": "possession_for",
    "possession_home": "possession_against",
    "rank_away": "team_rank",
    "rank_home": "opp_rank",
    "away_recruiting_avg_mean_4yr": "recruiting_avg_mean_4yr",
    "away_recruiting_points_mean_4yr": "recruiting_points_mean_4yr",
    "away_recruiting_points_sum_4yr": "recruiting_points_sum_4yr",
    "away_overall_avg": "overall_avg",
    "away_overall_points": "overall_points",
    "away_transfer_avg": "transfer_avg",
    "away_transfer_points": "transfer_points"
})
away_df["is_home"] = 0

## Concat Home & Away Long Tables

In [35]:
team_games = pd.concat([home_df, away_df], ignore_index=True)

team_games = team_games.sort_values(["team", "season", "date", "game_id"]).reset_index(drop=True)

team_games["won"] = (team_games["points_for"] > team_games["points_against"]).astype(int)
team_games["margin"] = team_games["points_for"] - team_games["points_against"]
team_games["conference_game"] = (team_games["team_conf"] == team_games["opp_conf"]).astype(int)

team_games["team_rank_filled"] = team_games["team_rank"].fillna(rank_fallback)
team_games["opp_rank_filled"] = team_games["opp_rank"].fillna(rank_fallback)
team_games["ranked_team"] = team_games["team_rank"].notna().astype(int)
team_games["ranked_opp"] = team_games["opp_rank"].notna().astype(int)

team_games

,game_id,season,week,date,team,opponent,team_conf,opp_conf,neutral,points_for,...,transfer_avg,transfer_points,is_home,won,margin,conference_game,team_rank_filled,opp_rank_filled,ranked_team,ranked_opp
0,8,2014,1.0,2014-08-30,alabama,west virginia,sec,big12,1,33,...,NaN,NaN,1,1,10,0,2.0,999.0,1,0
1,40,2014,4.0,2014-09-20,alabama,florida,sec,sec,0,42,...,NaN,NaN,1,1,21,1,3.0,999.0,1,0
2,79,2014,6.0,2014-10-04,alabama,ole miss,sec,sec,0,17,...,NaN,NaN,0,0,-6,1,3.0,11.0,1,1
3,106,2014,7.0,2014-10-11,alabama,arkansas,sec,sec,0,14,...,NaN,NaN,0,1,1,1,7.0,999.0,1,0
4,151,2014,8.0,2014-10-18,alabama,texas a&m,sec,sec,0,59,...,NaN,NaN,1,1,59,1,7.0,21.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7641,3683,2025,9.0,2025-10-25,wisconsin,oregon,big10,big10,0,7,...,87.24,47.4,0,0,-14,1,999.0,6.0,0,1
7642,3734,2025,11.0,2025-11-08,wisconsin,washington,big10,big10,0,13,...,87.24,47.4,1,1,3,1,999.0,23.0,0,1
7643,3760,2025,12.0,2025-11-15,wisconsin,indiana,big10,big10,0,7,...,87.24,47.4,0,0,-24,1,999.0,2.0,0,1
7644,3768,2025,13.0,2025-11-22,wisconsin,illinois,big10,big10,0,27,...,87.24,47.4,1,1,17,1,999.0,21.0,0,1


## Team-Game Level Rate/Efficiency Featues (Non-Rolled/lagged)

In [36]:
team_games["third_down_rate_for"] = safe_divide(team_games["third_down_comp_for"], team_games["third_down_att_for"])
team_games["third_down_rate_against"] = safe_divide(team_games["third_down_comp_against"], team_games["third_down_att_against"])

team_games["fourth_down_rate_for"] = safe_divide(team_games["fourth_down_comp_for"], team_games["fourth_down_att_for"])
team_games["fourth_down_rate_against"] = safe_divide(team_games["fourth_down_comp_against"], team_games["fourth_down_att_against"])

team_games["yards_per_pass_att_for"] = safe_divide(team_games["pass_yards_for"], team_games["pass_att_for"])
team_games["yards_per_pass_att_against"] = safe_divide(team_games["pass_yards_against"], team_games["pass_att_against"])

team_games["yards_per_rush_for"] = safe_divide(team_games["rush_yards_for"], team_games["rush_att_for"])
team_games["yards_per_rush_against"] = safe_divide(team_games["rush_yards_against"], team_games["rush_att_against"])

team_games["plays_for"] = team_games["pass_att_for"] + team_games["rush_att_for"]
team_games["plays_against"] = team_games["pass_att_against"] + team_games["rush_att_against"]

team_games["yards_per_play_for"] = safe_divide(team_games["total_yards_for"], team_games["plays_for"])
team_games["yards_per_play_against"] = safe_divide(team_games["total_yards_against"], team_games["plays_against"])

team_games["turnovers_for"] = team_games["fum_for"] + team_games["int_for"]
team_games["turnovers_against"] = team_games["fum_against"] + team_games["int_against"]
team_games["turnover_margin"] = team_games["turnovers_against"] - team_games["turnovers_for"]

team_games["pen_yards_per_pen_for"] = safe_divide(team_games["pen_yards_for"], team_games["pen_num_for"])
team_games["pen_yards_per_pen_against"] = safe_divide(team_games["pen_yards_against"], team_games["pen_num_against"])
team_games["pen_yards_per_play_for"] = safe_divide(team_games["pen_yards_for"], team_games["plays_for"])
team_games["pen_yards_per_play_against"] = safe_divide(team_games["pen_yards_against"], team_games["plays_against"])

team_games["points_per_play_for"] = safe_divide(team_games["points_for"], team_games["plays_for"])
team_games["points_per_play_against"] = safe_divide(team_games["points_against"], team_games["plays_against"])

team_games

,game_id,season,week,date,team,opponent,team_conf,opp_conf,neutral,points_for,...,yards_per_play_against,turnovers_for,turnovers_against,turnover_margin,pen_yards_per_pen_for,pen_yards_per_pen_against,pen_yards_per_play_for,pen_yards_per_play_against,points_per_play_for,points_per_play_against
0,8,2014,1.0,2014-08-30,alabama,west virginia,sec,big12,1,33,...,5.695652,1.0,0.0,-1.0,7.000000,9.166667,0.597561,0.797101,0.402439,0.333333
1,40,2014,4.0,2014-09-20,alabama,florida,sec,sec,0,42,...,3.636364,4.0,3.0,-1.0,7.272727,7.200000,0.919540,0.654545,0.482759,0.381818
2,79,2014,6.0,2014-10-04,alabama,ole miss,sec,sec,0,17,...,5.190476,2.0,1.0,-1.0,6.500000,8.333333,0.693333,0.396825,0.226667,0.365079
3,106,2014,7.0,2014-10-11,alabama,arkansas,sec,sec,0,14,...,4.240506,2.0,3.0,1.0,7.500000,7.000000,0.566038,0.354430,0.264151,0.164557
4,151,2014,8.0,2014-10-18,alabama,texas a&m,sec,sec,0,59,...,3.127273,0.0,1.0,1.0,NaN,3.000000,0.000000,0.109091,0.737500,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7641,3683,2025,9.0,2025-10-25,wisconsin,oregon,big10,big10,0,7,...,5.234375,1.0,0.0,-1.0,8.375000,8.333333,1.340000,1.171875,0.140000,0.328125
7642,3734,2025,11.0,2025-11-08,wisconsin,washington,big10,big10,0,13,...,3.676471,0.0,2.0,2.0,8.333333,6.625000,0.384615,0.779412,0.200000,0.147059
7643,3760,2025,12.0,2025-11-15,wisconsin,indiana,big10,big10,0,7,...,6.258065,2.0,0.0,-2.0,5.000000,10.000000,0.111111,0.161290,0.155556,0.500000
7644,3768,2025,13.0,2025-11-22,wisconsin,illinois,big10,big10,0,27,...,4.656250,0.0,0.0,0.0,10.000000,13.333333,0.847458,0.625000,0.457627,0.156250


## Roll/Lag Features to Create Pre-game Features

In [37]:
roll_cols = [
    "won",
    "margin",
    "points_for",
    "points_against",
    "first_downs_for",
    "first_downs_against",
    "third_down_rate_for",
    "third_down_rate_against",
    "fourth_down_rate_for",
    "fourth_down_rate_against",
    "pass_yards_for",
    "pass_yards_against",
    "rush_yards_for",
    "rush_yards_against",
    "total_yards_for",
    "total_yards_against",
    "yards_per_pass_att_for",
    "yards_per_pass_att_against",
    "yards_per_rush_for",
    "yards_per_rush_against",
    "yards_per_play_for",
    "yards_per_play_against",
    "turnovers_for",
    "turnovers_against",
    "turnover_margin",
    "pen_yards_for",
    "pen_yards_against",
    "pen_yards_per_play_for",
    "pen_yards_per_play_against",
    "points_per_play_for",
    "points_per_play_against",
    "possession_for",
    "possession_against"
]

grouped = team_games.groupby(["team", "season"], group_keys=False)

# lag-1 values
for col in roll_cols:
    team_games[f"{col}_lag1"] = grouped[col].shift(1)

# expanding means through prior games
for col in roll_cols:
    team_games[f"{col}_pre_mean"] = grouped[col].apply(lambda s: s.shift(1).expanding().mean())

# recent-form last 3 games
for col in roll_cols:
    team_games[f"{col}_pre_last3"] = grouped[col].apply(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

# count of prior games played in that season
team_games["games_played_pre"] = grouped.cumcount()

# prior win pct
team_games["win_pct_pre"] = team_games["won_pre_mean"]

# season-to-date rank summaries
team_games["team_rank_pre"] = grouped["team_rank_filled"].apply(lambda s: s.shift(1).expanding().mean())
team_games["ranked_team_pre"] = grouped["ranked_team"].apply(lambda s: s.shift(1).expanding().mean())

# static-ish roster variables copied as-is
static_pre_cols = [
    "recruiting_avg_mean_4yr",
    "recruiting_points_mean_4yr",
    "recruiting_points_sum_4yr",
    "overall_avg",
    "overall_points",
    "transfer_avg",
    "transfer_points"
]

# fill transfer/overall missingness with 0 and add indicators
for col in ["overall_avg", "overall_points", "transfer_avg", "transfer_points"]:
    team_games[f"{col}_missing"] = team_games[col].isna().astype(int)
    team_games[col] = team_games[col].fillna(0)

team_games

/var/folders/kk/54xn9lv90wq5vmw11pj1svvr0000gn/T/ipykernel_46231/2119487818.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_games[f"{col}_pre_last3"] = grouped[col].apply(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
/var/folders/kk/54xn9lv90wq5vmw11pj1svvr0000gn/T/ipykernel_46231/2119487818.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_games[f"{col}_pre_last3"] = grouped[col].apply(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
/var/folders/kk/54xn9lv90wq5vmw11pj1svvr0000gn/T/ipykernel_462

,game_id,season,week,date,team,opponent,team_conf,opp_conf,neutral,points_for,...,possession_for_pre_last3,possession_against_pre_last3,games_played_pre,win_pct_pre,team_rank_pre,ranked_team_pre,overall_avg_missing,overall_points_missing,transfer_avg_missing,transfer_points_missing
0,8,2014,1.0,2014-08-30,alabama,west virginia,sec,big12,1,33,...,NaN,NaN,0,NaN,NaN,NaN,1,1,1,1
1,40,2014,4.0,2014-09-20,alabama,florida,sec,sec,0,42,...,37.780000,22.220000,1,1.000000,2.000000,1.0,1,1,1,1
2,79,2014,6.0,2014-10-04,alabama,ole miss,sec,sec,0,17,...,38.525000,21.475000,2,1.000000,2.500000,1.0,1,1,1,1
3,106,2014,7.0,2014-10-11,alabama,arkansas,sec,sec,0,14,...,36.800000,23.190000,3,0.666667,2.666667,1.0,1,1,1,1
4,151,2014,8.0,2014-10-18,alabama,texas a&m,sec,sec,0,59,...,38.240000,27.190000,4,0.750000,3.750000,1.0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7641,3683,2025,9.0,2025-10-25,wisconsin,oregon,big10,big10,0,7,...,28.583333,31.416667,5,0.000000,999.000000,0.0,0,0,0,0
7642,3734,2025,11.0,2025-11-08,wisconsin,washington,big10,big10,0,13,...,26.616667,33.383333,6,0.000000,999.000000,0.0,0,0,0,0
7643,3760,2025,12.0,2025-11-15,wisconsin,indiana,big10,big10,0,7,...,27.490000,32.510000,7,0.142857,999.000000,0.0,0,0,0,0
7644,3768,2025,13.0,2025-11-22,wisconsin,illinois,big10,big10,0,27,...,27.483333,32.516667,8,0.125000,999.000000,0.0,0,0,0,0


## Retain Specified Feature to Merge Back Onto Team Games DF

In [38]:
pregame_cols = [
    "game_id", "team", "season", "week", "date", "is_home",
    "games_played_pre",
    "win_pct_pre",
    "team_rank_pre",
    "ranked_team_pre",
    "recruiting_avg_mean_4yr",
    "recruiting_points_mean_4yr",
    "recruiting_points_sum_4yr",
    "overall_avg",
    "overall_points",
    "transfer_avg",
    "transfer_points",
    "overall_avg_missing",
    "overall_points_missing",
    "transfer_avg_missing",
    "transfer_points_missing",

    "margin_pre_mean",
    "margin_pre_last3",
    "points_for_pre_mean",
    "points_against_pre_mean",
    "points_for_pre_last3",
    "points_against_pre_last3",

    "first_downs_for_pre_mean",
    "first_downs_against_pre_mean",
    "third_down_rate_for_pre_mean",
    "third_down_rate_against_pre_mean",
    "fourth_down_rate_for_pre_mean",
    "fourth_down_rate_against_pre_mean",

    "pass_yards_for_pre_mean",
    "pass_yards_against_pre_mean",
    "rush_yards_for_pre_mean",
    "rush_yards_against_pre_mean",
    "total_yards_for_pre_mean",
    "total_yards_against_pre_mean",

    "yards_per_pass_att_for_pre_mean",
    "yards_per_pass_att_against_pre_mean",
    "yards_per_rush_for_pre_mean",
    "yards_per_rush_against_pre_mean",
    "yards_per_play_for_pre_mean",
    "yards_per_play_against_pre_mean",

    "turnovers_for_pre_mean",
    "turnovers_against_pre_mean",
    "turnover_margin_pre_mean",

    "pen_yards_for_pre_mean",
    "pen_yards_against_pre_mean",
    "pen_yards_per_play_for_pre_mean",
    "pen_yards_per_play_against_pre_mean",

    "points_per_play_for_pre_mean",
    "points_per_play_against_pre_mean",

    "possession_for_pre_mean",
    "possession_against_pre_mean"
]

pregame = team_games[pregame_cols].copy()

# Merge Home and Away Pre-game Features onto Game Table

In [39]:
home_pre = pregame.loc[pregame["is_home"] == 1].copy()
away_pre = pregame.loc[pregame["is_home"] == 0].copy()

home_pre = home_pre.drop(columns=["is_home"]).add_prefix("home_pre_")
away_pre = away_pre.drop(columns=["is_home"]).add_prefix("away_pre_")

model_df = df.copy()

model_df = model_df.merge(
    home_pre,
    left_on=["game_id", "home", "season", "week", "date"],
    right_on=["home_pre_game_id", "home_pre_team", "home_pre_season", "home_pre_week", "home_pre_date"],
    how="left"
)

model_df = model_df.merge(
    away_pre,
    left_on=["game_id", "away", "season", "week", "date"],
    right_on=["away_pre_game_id", "away_pre_team", "away_pre_season", "away_pre_week", "away_pre_date"],
    how="left"
)
model_df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,away_pre_turnovers_against_pre_mean,away_pre_turnover_margin_pre_mean,away_pre_pen_yards_for_pre_mean,away_pre_pen_yards_against_pre_mean,away_pre_pen_yards_per_play_for_pre_mean,away_pre_pen_yards_per_play_against_pre_mean,away_pre_points_per_play_for_pre_mean,away_pre_points_per_play_against_pre_mean,away_pre_possession_for_pre_mean,away_pre_possession_against_pre_mean
0,2014,1.0,2014-08-28,10:00 PM,rutgers,washington state,NaN,NaN,big10,pac12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014,1.0,2014-08-28,6:00 PM,texas a&m,south carolina,21.0,9.0,sec,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014,1.0,2014-08-30,4:00 PM,arkansas,auburn,NaN,6.0,sec,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014,1.0,2014-08-30,3:30 PM,california,northwestern,NaN,NaN,pac12,big10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014,1.0,2014-08-30,5:30 PM,clemson,georgia,16.0,12.0,acc,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,3:30 PM,wisconsin,minnesota,NaN,NaN,big10,big10,...,0.333333,-0.888889,34.666667,37.222222,0.629211,0.607191,0.170333,0.423917,29.793333,30.206667
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,1.900000,1.000000,40.000000,45.100000,0.590539,0.711263,0.445114,0.302843,33.262000,26.738000
3820,2025,15.0,2025-12-06,8:00 PM,duke,virginia,NaN,17.0,acc,acc,...,2.000000,1.222222,64.777778,62.444444,0.981968,0.942776,0.511560,0.433467,29.705556,30.294444
3821,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,0.888889,-0.111111,48.555556,53.555556,0.713093,0.859784,0.434996,0.317930,33.376667,26.623333


## Make Matchup-Difference features
Home - Away

In [40]:
diff_pairs = [
    ("games_played_pre", "games_played_diff"),
    ("win_pct_pre", "win_pct_diff"),
    ("team_rank_pre", "rank_pre_diff"),      # smaller is better, so negative means home better ranked
    ("ranked_team_pre", "ranked_rate_diff"),

    ("recruiting_avg_mean_4yr", "recruiting_avg_4yr_diff"),
    ("recruiting_points_mean_4yr", "recruiting_points_mean_4yr_diff"),
    ("recruiting_points_sum_4yr", "recruiting_points_sum_4yr_diff"),
    ("overall_avg", "overall_avg_diff"),
    ("overall_points", "overall_points_diff"),
    ("transfer_avg", "transfer_avg_diff"),
    ("transfer_points", "transfer_points_diff"),

    ("margin_pre_mean", "margin_pre_mean_diff"),
    ("margin_pre_last3", "margin_pre_last3_diff"),
    ("points_for_pre_mean", "points_for_pre_mean_diff"),
    ("points_against_pre_mean", "points_against_pre_mean_diff"),
    ("points_for_pre_last3", "points_for_pre_last3_diff"),
    ("points_against_pre_last3", "points_against_pre_last3_diff"),

    ("first_downs_for_pre_mean", "first_downs_for_pre_mean_diff"),
    ("first_downs_against_pre_mean", "first_downs_against_pre_mean_diff"),

    ("third_down_rate_for_pre_mean", "third_down_rate_for_pre_mean_diff"),
    ("third_down_rate_against_pre_mean", "third_down_rate_against_pre_mean_diff"),
    ("fourth_down_rate_for_pre_mean", "fourth_down_rate_for_pre_mean_diff"),
    ("fourth_down_rate_against_pre_mean", "fourth_down_rate_against_pre_mean_diff"),

    ("pass_yards_for_pre_mean", "pass_yards_for_pre_mean_diff"),
    ("pass_yards_against_pre_mean", "pass_yards_against_pre_mean_diff"),
    ("rush_yards_for_pre_mean", "rush_yards_for_pre_mean_diff"),
    ("rush_yards_against_pre_mean", "rush_yards_against_pre_mean_diff"),
    ("total_yards_for_pre_mean", "total_yards_for_pre_mean_diff"),
    ("total_yards_against_pre_mean", "total_yards_against_pre_mean_diff"),

    ("yards_per_pass_att_for_pre_mean", "yppa_for_pre_mean_diff"),
    ("yards_per_pass_att_against_pre_mean", "yppa_against_pre_mean_diff"),
    ("yards_per_rush_for_pre_mean", "ypra_for_pre_mean_diff"),
    ("yards_per_rush_against_pre_mean", "ypra_against_pre_mean_diff"),
    ("yards_per_play_for_pre_mean", "ypp_for_pre_mean_diff"),
    ("yards_per_play_against_pre_mean", "ypp_against_pre_mean_diff"),

    ("turnovers_for_pre_mean", "turnovers_for_pre_mean_diff"),
    ("turnovers_against_pre_mean", "turnovers_against_pre_mean_diff"),
    ("turnover_margin_pre_mean", "turnover_margin_pre_mean_diff"),

    ("pen_yards_for_pre_mean", "pen_yards_for_pre_mean_diff"),
    ("pen_yards_against_pre_mean", "pen_yards_against_pre_mean_diff"),
    ("pen_yards_per_play_for_pre_mean", "pen_yards_per_play_for_pre_mean_diff"),
    ("pen_yards_per_play_against_pre_mean", "pen_yards_per_play_against_pre_mean_diff"),

    ("points_per_play_for_pre_mean", "ppp_for_pre_mean_diff"),
    ("points_per_play_against_pre_mean", "ppp_against_pre_mean_diff"),

    ("possession_for_pre_mean", "possession_for_pre_mean_diff"),
    ("possession_against_pre_mean", "possession_against_pre_mean_diff"),
]

for base_name, diff_name in diff_pairs:
    model_df[diff_name] = model_df[f"home_pre_{base_name}"] - model_df[f"away_pre_{base_name}"]

# offense vs defense matchup style features
model_df["pass_matchup_diff"] = (
    model_df["home_pre_pass_yards_for_pre_mean"] - model_df["away_pre_pass_yards_against_pre_mean"]
)
model_df["rush_matchup_diff"] = (
    model_df["home_pre_rush_yards_for_pre_mean"] - model_df["away_pre_rush_yards_against_pre_mean"]
)
model_df["ypp_matchup_diff"] = (
    model_df["home_pre_yards_per_play_for_pre_mean"] - model_df["away_pre_yards_per_play_against_pre_mean"]
)

model_df["opp_pass_matchup_diff"] = (
    model_df["away_pre_pass_yards_for_pre_mean"] - model_df["home_pre_pass_yards_against_pre_mean"]
)
model_df["opp_rush_matchup_diff"] = (
    model_df["away_pre_rush_yards_for_pre_mean"] - model_df["home_pre_rush_yards_against_pre_mean"]
)
model_df["opp_ypp_matchup_diff"] = (
    model_df["away_pre_yards_per_play_for_pre_mean"] - model_df["home_pre_yards_per_play_against_pre_mean"]
)

# rank gap = positive means home better ranked
model_df["rank_advantage_home"] = model_df["rank_away_filled"] - model_df["rank_home_filled"]
model_df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,ppp_against_pre_mean_diff,possession_for_pre_mean_diff,possession_against_pre_mean_diff,pass_matchup_diff,rush_matchup_diff,ypp_matchup_diff,opp_pass_matchup_diff,opp_rush_matchup_diff,opp_ypp_matchup_diff,rank_advantage_home
0,2014,1.0,2014-08-28,10:00 PM,rutgers,washington state,NaN,NaN,big10,pac12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,2014,1.0,2014-08-28,6:00 PM,texas a&m,south carolina,21.0,9.0,sec,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0
2,2014,1.0,2014-08-30,4:00 PM,arkansas,auburn,NaN,6.0,sec,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,993.0
3,2014,1.0,2014-08-30,3:30 PM,california,northwestern,NaN,NaN,pac12,big10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,2014,1.0,2014-08-30,5:30 PM,clemson,georgia,16.0,12.0,acc,sec,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,3:30 PM,wisconsin,minnesota,NaN,NaN,big10,big10,...,0.032490,-1.980000,1.980000,-58.000000,-41.111111,-1.295064,-150.333333,-34.777778,-2.261528,0.0
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,-0.118937,-2.177556,2.177556,74.066667,58.711111,0.925399,13.222222,100.955556,1.543961,7.0
3820,2025,15.0,2025-12-06,8:00 PM,duke,virginia,NaN,17.0,acc,acc,...,-0.097237,1.964444,-1.964444,-55.033333,36.877778,-0.702411,75.711111,9.433333,1.063331,982.0
3821,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,0.011817,-0.810667,0.810667,42.355556,6.522222,0.498648,37.100000,46.388889,0.473197,-6.0


## Filter model_df; Keep games with >= 1 prior game for both teams

In [41]:
model_df["both_teams_have_prior_game"] = (
    (model_df["home_pre_games_played_pre"] >= 1) &
    (model_df["away_pre_games_played_pre"] >= 1)
).astype(int)

# choose whether to keep only games where both teams have prior-game history
model_df = model_df.loc[model_df["both_teams_have_prior_game"] == 1].copy()

model_df

,season,week,date,time_et,away,home,rank_away,rank_home,conf_away,conf_home,...,possession_for_pre_mean_diff,possession_against_pre_mean_diff,pass_matchup_diff,rush_matchup_diff,ypp_matchup_diff,opp_pass_matchup_diff,opp_rush_matchup_diff,opp_ypp_matchup_diff,rank_advantage_home,both_teams_have_prior_game
22,2014,3.0,2014-09-13,3:30 PM,georgia,south carolina,6.0,24.0,sec,sec,...,-7.680000,7.680000,163.000000,-12.000000,3.154924,-380.000000,159.000000,-0.017941,-18.0,1
27,2014,3.0,2014-09-13,12:30 PM,louisville,virginia,21.0,NaN,acc,acc,...,1.450000,-1.220000,92.000000,50.000000,0.238095,-36.000000,14.000000,-0.171715,-978.0,1
29,2014,3.0,2014-09-13,8:00 PM,penn state,rutgers,NaN,NaN,big10,big10,...,-0.420000,1.270000,59.000000,191.000000,2.262185,-78.000000,51.000000,-0.872381,0.0,1
33,2014,3.0,2014-09-13,8:15 PM,ucla,texas,12.0,NaN,pac12,big12,...,-0.150000,-0.380000,-90.000000,-38.000000,-0.686147,61.000000,-132.000000,0.029110,-987.0,1
34,2014,3.0,2014-09-13,8:00 PM,usc,boston college,9.0,NaN,pac12,acc,...,0.610000,-0.210000,-151.000000,14.000000,-1.621917,24.000000,-147.000000,-0.898783,-990.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,3:30 PM,wisconsin,minnesota,NaN,NaN,big10,big10,...,-1.980000,1.980000,-58.000000,-41.111111,-1.295064,-150.333333,-34.777778,-2.261528,0.0,1
3819,2025,15.0,2025-12-06,12:00 PM,byu,texas tech,11.0,4.0,big12,big12,...,-2.177556,2.177556,74.066667,58.711111,0.925399,13.222222,100.955556,1.543961,7.0,1
3820,2025,15.0,2025-12-06,8:00 PM,duke,virginia,NaN,17.0,acc,acc,...,1.964444,-1.964444,-55.033333,36.877778,-0.702411,75.711111,9.433333,1.063331,982.0,1
3821,2025,15.0,2025-12-06,4:00 PM,georgia,alabama,3.0,9.0,sec,sec,...,-0.810667,0.810667,42.355556,6.522222,0.498648,37.100000,46.388889,0.473197,-6.0,1


## Official Model Table

In [42]:
base_feature_cols = [
    "season", "week", "date", "away", "home",
    "neutral", "conference_game", "ranked_home", "ranked_away", "both_ranked",
    "margin_home", "home_win",

    "home_pre_games_played_pre",
    "away_pre_games_played_pre",

    "rank_advantage_home",
    "win_pct_diff",
    "margin_pre_mean_diff",
    "margin_pre_last3_diff",

    "recruiting_avg_4yr_diff",
    "recruiting_points_mean_4yr_diff",
    "overall_avg_diff",
    "overall_points_diff",
    "transfer_avg_diff",
    "transfer_points_diff",

    "points_for_pre_mean_diff",
    "points_against_pre_mean_diff",
    "ypp_for_pre_mean_diff",
    "ypp_against_pre_mean_diff",
    "yppa_for_pre_mean_diff",
    "ypra_for_pre_mean_diff",
    "turnover_margin_pre_mean_diff",
    "third_down_rate_for_pre_mean_diff",
    "third_down_rate_against_pre_mean_diff",
    "possession_for_pre_mean_diff",

    "pass_matchup_diff",
    "rush_matchup_diff",
    "ypp_matchup_diff",
    "opp_pass_matchup_diff",
    "opp_rush_matchup_diff",
    "opp_ypp_matchup_diff",

    "home_pre_overall_avg_missing",
    "home_pre_transfer_avg_missing",
    "away_pre_overall_avg_missing",
    "away_pre_transfer_avg_missing",
]

model_table = model_df[base_feature_cols].copy()

model_table["avg_games_played_pre"] = (model_table["home_pre_games_played_pre"] + model_table["away_pre_games_played_pre"]) / 2

model_table["season_phase"] = pd.cut(
    model_table["avg_games_played_pre"],
    bins=[-np.inf, 2, 6, np.inf],
    labels=["early", "mid", "late"]
)

model_table['rank_status'] = np.select(
    [
        model_table['rank_advantage_home'] > 0,
        model_table['rank_advantage_home'] < 0,
        model_table['rank_advantage_home'] == 0
    ],
    [
        'home_higher_ranked',
        'away_higher_ranked',
        'both_unranked'
    ],
    default='other'
)

print(model_table.shape)

model_table.to_csv(project_data_exports_path / 'model_table.csv', index=False)
model_table

(3304, 47)


,season,week,date,away,home,neutral,conference_game,ranked_home,ranked_away,both_ranked,...,opp_pass_matchup_diff,opp_rush_matchup_diff,opp_ypp_matchup_diff,home_pre_overall_avg_missing,home_pre_transfer_avg_missing,away_pre_overall_avg_missing,away_pre_transfer_avg_missing,avg_games_played_pre,season_phase,rank_status
22,2014,3.0,2014-09-13,georgia,south carolina,0,1,1,1,1,...,-380.000000,159.000000,-0.017941,1,1,1,1,1.0,early,away_higher_ranked
27,2014,3.0,2014-09-13,louisville,virginia,0,1,0,1,0,...,-36.000000,14.000000,-0.171715,1,1,1,1,1.0,early,away_higher_ranked
29,2014,3.0,2014-09-13,penn state,rutgers,0,1,0,0,0,...,-78.000000,51.000000,-0.872381,1,1,1,1,1.0,early,both_unranked
33,2014,3.0,2014-09-13,ucla,texas,1,0,0,1,0,...,61.000000,-132.000000,0.029110,1,1,1,1,1.0,early,away_higher_ranked
34,2014,3.0,2014-09-13,usc,boston college,0,0,0,1,0,...,24.000000,-147.000000,-0.898783,1,1,1,1,1.0,early,away_higher_ranked
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2025,14.0,2025-11-29,wisconsin,minnesota,0,1,0,0,0,...,-150.333333,-34.777778,-2.261528,0,0,0,0,9.0,late,both_unranked
3819,2025,15.0,2025-12-06,byu,texas tech,1,1,1,1,1,...,13.222222,100.955556,1.543961,0,0,0,0,9.5,late,home_higher_ranked
3820,2025,15.0,2025-12-06,duke,virginia,1,1,1,0,0,...,75.711111,9.433333,1.063331,0,0,0,0,9.5,late,home_higher_ranked
3821,2025,15.0,2025-12-06,georgia,alabama,1,1,1,1,1,...,37.100000,46.388889,0.473197,0,0,0,0,9.5,late,away_higher_ranked
